# 点时申万行业数据准备

本 Notebook 将项目根目录 `swind/` 下按交易日分片的 UTF-8 CSV，标准化并与 `data/processed/daily_clean.parquet` 的 `(trade_date, stock_code)` 键做左连接。输出保留申万一、二、三级代码和名称，但阶段三候选因子行业中性化只使用当日 `sw_code_1`。

全量源数据约 1.49 GB。检查和转换均由你显式开启；代码不会修改原 CSV、基础行情或六特征张量。

In [ ]:
# 1. 环境、路径与配置
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name.lower() == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from factor_gfn.data.industry import (
    IndustryBuildConfig,
    build_sw_industry_daily,
    inspect_sw_industry_output,
    inspect_sw_industry_source,
    load_sw_industry_panel,
)

config = IndustryBuildConfig()
print('CSV目录:', config.source_dir)
print('行情键:', config.market_keys_path)
print('输出长表:', config.output_path)
print('输出元数据:', config.metadata_path)

## 2. 可选的只读全量检查

该单元会扫描全部 CSV，但不写输出。正式构建函数也会执行同一套严格检查，因此如果准备直接构建，可以跳过此单元，避免重复扫描。

In [ ]:
RUN_SOURCE_INSPECTION = False  # 只想先看源数据 QA 时改为 True

if RUN_SOURCE_INSPECTION:
    source_summary = inspect_sw_industry_source(config)
    display(source_summary)
else:
    print('未执行全量源数据检查。正式构建时会自动检查。')

## 3. 全量构建

开启后会严格校验表头、文件日期、股票后缀、三级行业代码和重复键，再通过 DuckDB 流式生成长表。输出必须与 `daily_clean.parquet` 的键和行数完全一致；异常时不会覆盖已有正式文件。

In [ ]:
RUN_FULL_BUILD = True  # 确认路径后手动改为 True

if RUN_FULL_BUILD:
    industry_metadata = build_sw_industry_daily(config)
    display(industry_metadata['source_summary'])
    display(industry_metadata['output_summary'])
else:
    print('未执行 1.49 GB 全量转换。')

## 4. 输出 QA 与小范围面板对齐

构建完成后，该单元展示覆盖率和最差日期，并只读取前 5 个日期、前 10 只股票验证 `(date, stock)` 一级行业代码矩阵。矩阵使用 `-1` 表示行业缺失。

In [ ]:
if config.output_path.exists():
    output_summary = inspect_sw_industry_output(config.output_path)
    display(output_summary)

    dates = np.load(project_root / 'data/processed/date_list.npy', allow_pickle=False)[:5]
    stocks = np.load(project_root / 'data/processed/stock_list.npy', allow_pickle=False)[:10]
    level1_panel = load_sw_industry_panel(dates, stocks, level=1, path=config.output_path)
    display(pd.DataFrame(level1_panel, index=dates.astype(str), columns=stocks))
else:
    print('尚未生成行业长表，请先确认并运行全量构建单元。')